# Part A: Data Foundation

**Goal:** Clean messy data and load it into a database.

**Key techniques:** handle missing values, outliers and duplicates; standardize city and gender; SQLite.

## How to use this notebook
1. Keep this notebook in the **project root** (next to `generate_data.py`, `src/`, `data/`). It also works from a sub-folder; the setup cell finds the root.
2. Select the project's `.venv` as the kernel (top-right of the notebook in VS Code).
3. Run the cells **top to bottom** (`Run All`). Later cells depend on earlier ones.

## What is inside
The code in each section is the code from your `.py` file, split into cells with no change to the logic. Sections run in this order:

1. `check_raw_data.py` - look at the raw files
2. `clean_data.py` - clean and save CSVs + SQLite
3. `validate_data.py` - assert the cleaned data is correct
4. `check_database.py` - inspect the SQLite tables
5. `run_sql_queries.py` - run `sql/queries.sql`
6. `eda.py` - exploratory data analysis

The original `.py` files are untouched and are still needed: some sections import each other (for example `from policy_rag import ...`) and `mock_llm.py` is the supplied local LLM.


In [ ]:
# Setup cell: lets the original script code run unchanged inside a notebook.
# 1) finds the project root, 2) makes it the working directory (the scripts use
# relative paths like "data/raw"), 3) makes the project's own modules importable,
# 4) defines __file__ (scripts use it, notebooks do not have it).
import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "src").is_dir() and (candidate / "mock_llm.py").exists():
        ROOT = candidate
        break

os.chdir(ROOT)
for p in (str(ROOT), str(ROOT / "src")):
    if p not in sys.path:
        sys.path.insert(0, p)

__file__ = str(ROOT / "src" / "notebook.py")
print("Project root:", ROOT)

---
## `src/check_raw_data.py`
**Why:** never clean data you have not looked at. This prints shape, first rows, data types and missing values of each raw file.

**Original purpose note from the script:**

> Purpose: To inspect raw data before cleaning.
> - Reads customers.csv, products.csv, orders.csv, and tickets.csv.
> - Displays the number of rows and columns.
> - Shows the first three records.
> - Displays column data types and missing values.

In [ ]:
from pathlib import Path
import pandas as pd
data_dir = Path("data/raw")
files = [
    "customers.csv",
    "products.csv",
    "orders.csv",
    "tickets.csv",
]
for filename in files:
    file_path = data_dir / filename

    print(f"\n{'=' * 40}")
    print(filename)

    df = pd.read_csv(file_path)

    print("Shape:", df.shape)
    print("\nFirst 3 rows:")
    print(df.head(3))
    print("\nColumn data types:")
    print(df.dtypes)
    print("\nMissing values:")
    print(df.isnull().sum())

---
## `src/clean_data.py`
**Why:** raw data has duplicates, bad amounts, negative delivery times, inconsistent city/gender spellings and missing ages. **How:** standardize text, turn invalid values into missing, fill with medians or catalogue price, drop duplicates, then save to `data/clean/` and `data/instamart.db`.

**Original purpose note from the script:**

> Purpose: To clean the data and store it for analysis.
> - Standardizes customer city and gender values.
> - Handles invalid or missing customer ages.
> - Converts dates and amounts into appropriate data types.
> - Removes duplicate orders and keeps the latest record for each order ID.
> - Fills missing order amounts and eligible delivery times.
> - Saves cleaned CSV files and creates SQLite database tables.
> - Prints a summary of the cleaned data.

In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd
# Project folders
RAW_DIR = Path("data/raw")
CLEAN_DIR = Path("data/clean")
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

### 1. Clean Customers

In [ ]:
# --------------------------------------------------
# 1. CLEAN CUSTOMERS
# --------------------------------------------------
customers = pd.read_csv(RAW_DIR / "customers.csv")
# Standardize city names
customers["city"] = customers["city"].astype("string").str.strip().str.title()
# Standardize gender values
gender_map = {
    "m": "male",
    "male": "male",
    "man": "male",
    "f": "female",
    "female": "female",
    "woman": "female",
}
gender_normalized = (
    customers["gender"].astype("string").str.strip().str.lower()
)
customers["gender"] = gender_normalized.map(gender_map).fillna("unknown")
# Convert age to numeric and treat ages outside 10-100 as missing
customers["age"] = pd.to_numeric(customers["age"], errors="coerce")
customers.loc[
    ~customers["age"].between(10, 100), "age"
] = float("nan")
# Fill missing ages using the median age within each city
city_medians = customers.groupby("city")["age"].transform("median")
customers["age"] = customers["age"].fillna(city_medians)
# Convert signup date
customers["signup_date"] = pd.to_datetime(
    customers["signup_date"], errors="coerce"
)

### 2. Load Products

In [ ]:
# --------------------------------------------------
# 2. LOAD PRODUCTS
# --------------------------------------------------
products = pd.read_csv(RAW_DIR / "products.csv")
products["price_inr"] = pd.to_numeric(
    products["price_inr"], errors="raise"
)

### 3. Clean Orders

In [ ]:
# --------------------------------------------------
# 3. CLEAN ORDERS
# --------------------------------------------------
orders = pd.read_csv(RAW_DIR / "orders.csv")
# i. Convert order timestamp
orders["order_ts"] = pd.to_datetime(
    orders["order_ts"], errors="coerce"
)
# ii. Remove exact duplicate rows
before_duplicates = len(orders)
orders = orders.drop_duplicates().copy()
duplicates_removed = before_duplicates - len(orders)
# iii. Keep only the latest record for each order ID
orders = orders.sort_values("order_ts")
orders = orders.drop_duplicates(
    subset=["order_id"], keep="last"
).copy()
# iv. Convert amounts to numbers
orders["order_amount"] = pd.to_numeric(
    orders["order_amount"], errors="coerce"
)
# Get the catalogue price and category for each product
orders = orders.merge(
    products[["product_id", "price_inr", "category"]],
    on="product_id",
    how="left",
    validate="many_to_one",
)
# Fill missing amounts using price, quantity and discount
calculated_amount = (
    orders["price_inr"]
    * orders["quantity"]
    * (1 - orders["discount_pct"] / 100)
).round(2)

In [ ]:
orders["order_amount"] = orders["order_amount"].fillna(
    calculated_amount
)
# v. Treat negative delivery times as missing
orders["delivery_minutes"] = pd.to_numeric(
    orders["delivery_minutes"], errors="coerce"
)
orders.loc[
    orders["delivery_minutes"] < 0, "delivery_minutes"
] = float("nan")
# Use completed orders to calculate category median delivery times
completed_mask = orders["status"].str.lower().eq("completed")
category_medians = (
    orders.loc[completed_mask]
    .groupby("category")["delivery_minutes"]
    .transform("median")
)
# Align category medians to the original order index
completed_orders = orders.loc[completed_mask].copy()
completed_orders["category_median"] = (
    completed_orders.groupby("category")["delivery_minutes"]
    .transform("median")
)
# Fill missing delivery times for completed orders only
orders.loc[completed_mask, "delivery_minutes"] = (
    orders.loc[completed_mask, "delivery_minutes"].fillna(
        completed_orders["category_median"]
    )
)
# Cancelled orders retain missing delivery times.
# Remove the temporary product lookup columns from orders.
orders = orders.drop(columns=["price_inr", "category"])

### 4. Save Clean Csv Files

In [ ]:
# --------------------------------------------------
# 4. SAVE CLEAN CSV FILES
# --------------------------------------------------
customers.to_csv(CLEAN_DIR / "customers.csv", index=False)
products.to_csv(CLEAN_DIR / "products.csv", index=False)
# Store timestamps in the requested text format
orders_to_save = orders.copy()
orders_to_save["order_ts"] = orders_to_save["order_ts"].dt.strftime(
    "%Y-%m-%d %H:%M:%S"
)
orders_to_save.to_csv(CLEAN_DIR / "orders.csv", index=False)

### 5. Save Tables To Sqlite

In [ ]:
# --------------------------------------------------
# 5. SAVE TABLES TO SQLITE
# --------------------------------------------------
database_path = Path("data/instamart.db")
with sqlite3.connect(database_path) as connection:
    customers.to_sql("customers", connection, if_exists="replace", index=False)
    products.to_sql("products", connection, if_exists="replace", index=False)
    orders_to_save.to_sql("orders", connection, if_exists="replace", index=False)

### 6. Print Summary

In [ ]:
# --------------------------------------------------
# 6. PRINT SUMMARY
# --------------------------------------------------
print("Customer rows:", len(customers))
print("Product rows:", len(products))
print("Order rows:", len(orders))
print("Exact duplicate rows removed:", duplicates_removed)
print("Order status counts:")
print(orders["status"].value_counts().to_string())
print("Missing customer ages:", customers["age"].isna().sum())
print("Missing order amounts:", orders["order_amount"].isna().sum())
print(
    "Missing delivery times among completed orders:",
    orders.loc[completed_mask, "delivery_minutes"].isna().sum(),
)
print(
    "Missing delivery times among cancelled orders:",
    orders.loc[
        orders["status"].str.lower().eq("cancelled"),
        "delivery_minutes",
    ].isna().sum(),
)
completed_revenue = orders.loc[
    completed_mask, "order_amount"
].sum()
print(f"Completed revenue: INR {completed_revenue:,.2f}")
print("Clean CSV files and SQLite database saved.")

---
## `src/validate_data.py`
**Why:** proves the cleaning worked. Each `assert` stops the run if a number is wrong; success prints `ALL CHECKPOINTS PASSED`.

**Original purpose note from the script:**

> Purpose: To verify the cleaned data and database.
> - Checks customer, product, and order row counts.
> - Verifies order ID uniqueness.
> - Checks customer ages, city names, and gender counts.
> - Validates order statuses, missing values, and total revenue.
> - Checks that the required SQLite tables contain the expected number of rows.
> - Prints ALL CHECKPOINTS PASSED if all assertions succeed.

In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd
# Load cleaned CSV files
customers = pd.read_csv("data/clean/customers.csv")
products = pd.read_csv("data/clean/products.csv")
orders = pd.read_csv("data/clean/orders.csv")
# Check row counts and order uniqueness
assert len(customers) == 1200, "Customer row count mismatch"
assert len(products) == 40, "Product row count mismatch"
assert len(orders) == 12309, "Order row count mismatch"
assert orders["order_id"].nunique() == 12309, "Duplicate order IDs found"
# Check customer cleaning
assert customers["age"].isna().sum() == 0, "Missing customer ages found"
assert customers["age"].between(18, 60).all(), "Age range mismatch"
assert set(customers["city"].unique()) == {
    "Pune", "Delhi", "Bengaluru",
    "Chennai", "Hyderabad", "Mumbai"
}, "City names mismatch"
assert customers["gender"].value_counts().to_dict() == {
    "female": 610,
    "male": 560,
    "unknown": 30,
}, "Gender counts mismatch"
# Check order statuses
status_counts = orders["status"].str.lower().value_counts().to_dict()
assert status_counts.get("completed", 0) == 11110, "Completed count mismatch"
assert status_counts.get("cancelled", 0) == 1199, "Cancelled count mismatch"
assert "pending" not in status_counts, "Pending orders remain"
# Check missing values and revenue
assert orders["order_amount"].notna().all(), "Missing order amounts found"

In [ ]:
completed = orders["status"].str.lower().eq("completed")
cancelled = orders["status"].str.lower().eq("cancelled")
assert orders.loc[completed, "delivery_minutes"].notna().all(), (
    "Completed orders have missing delivery times"
)
assert orders.loc[cancelled, "delivery_minutes"].isna().all(), (
    "Cancelled orders should have missing delivery times"
)
revenue = orders.loc[completed, "order_amount"].sum()
assert abs(revenue - 2998668.80) < 0.01, "Revenue mismatch"
# Check SQLite tables and row counts
with sqlite3.connect("data/instamart.db") as connection:
    tables = pd.read_sql_query(
        "SELECT name FROM sqlite_master WHERE type='table'",
        connection,
    )["name"].tolist()

    assert {"customers", "products", "orders"}.issubset(set(tables)), (
        "Required database tables are missing"
    )

    db_customers = pd.read_sql_query(
        "SELECT COUNT(*) AS n FROM customers", connection
    ).iloc[0]["n"]

    db_products = pd.read_sql_query(
        "SELECT COUNT(*) AS n FROM products", connection
    ).iloc[0]["n"]

    db_orders = pd.read_sql_query(
        "SELECT COUNT(*) AS n FROM orders", connection
    ).iloc[0]["n"]

    assert db_customers == 1200, "Database customer count mismatch"
    assert db_products == 40, "Database product count mismatch"
    assert db_orders == 12309, "Database order count mismatch"

In [ ]:
print("ALL CHECKPOINTS PASSED")

---
## `src/check_database.py`
**Why:** confirms the SQLite database exists and shows each table's schema.

In [ ]:
import sqlite3
from pathlib import Path
# Go up one level from src to the project root
PROJECT_ROOT = Path(__file__).resolve().parent.parent
DB_PATH = PROJECT_ROOT / "data" / "instamart.db"
print("Project root:", PROJECT_ROOT)
print("Database path:", DB_PATH)
print("Database exists:", DB_PATH.exists())
if not DB_PATH.exists():
    raise FileNotFoundError(f"Database not found: {DB_PATH}")
connection = sqlite3.connect(DB_PATH)
cursor = connection.cursor()
tables = cursor.execute("""
    SELECT name, sql
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name
""").fetchall()
print("\nTables and schemas:")
for table_name, schema in tables:
    print(f"\nTable: {table_name}")
    print(schema)
connection.close()

---
## `src/run_sql_queries.py`
**Why:** answers business questions (revenue by city/category, etc.) using SQL on the database.

**Original purpose note from the script:**

> Purpose: To execute SQL queries and display business insights.
> - Reads SQL queries from sql/queries.sql.
> - Connects to the SQLite database.
> - Executes each query.
> - Displays query results and selected calculations, such as total revenue and customer spending rankings.

In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd
database_path = Path("data/instamart.db")
sql_path = Path("sql/queries.sql")
sql_text = sql_path.read_text(encoding="utf-8")
queries = [
    query.strip()
    for query in sql_text.split(";")
    if query.strip()
]
with sqlite3.connect(database_path) as connection:
    for number, query in enumerate(queries, start=1):
        result = pd.read_sql_query(query, connection)

        print("\n" + "=" * 60)
        print(f"Q{number} — {len(result)} rows")
        print("=" * 60)

        if number == 2:
            print(result.to_string(index=False))
            print(
                "\nTotal revenue:",
                f"INR {result['revenue'].sum():,.2f}"
            )
        elif number == 6:
            print(
                result.sort_values(
                    ["city", "spend_rank", "customer_id"]
                ).groupby("city").head(1).to_string(index=False)
            )
            ties = (result["spend_rank"] != result["row_num"]).sum()
            print("Rows where RANK differs from ROW_NUMBER:", ties)
        elif number == 7:
            print(result.head(5).to_string(index=False))
            if not result.empty:
                print(
                    "Final cumulative revenue:",
                    f"INR {result['cumulative_revenue'].iloc[-1]:,.2f}"
                )
        else:
            print(result.head(10).to_string(index=False))

In [ ]:
print("\nAll seven SQL queries executed.")

---
## `src/eda.py`
**Why:** exploratory data analysis summarizes the clean data to understand patterns before modelling.

**Original purpose note from the script:**

> Purpose: To perform Exploratory Data Analysis (EDA) on cleaned Instamart data and discover useful business insights.
> EDA means examining data to understand patterns, revenue, relationships, and unusual values before building machine-learning models.
> What each section does
> Section	Purpose
> 1. Load data	           : Reads cleaned orders, customers, and products CSV files using pandas.
> 2. Data shapes	       : Displays the number of rows and columns in each dataset.
> 3. Revenue by city	   : Calculates total revenue, average order amount, and order count for each city, considering completed orders only.
> 4. Revenue by category  : Calculates total revenue and order count for each product category.
> 5. Order statistics	   : Calculates mean, median, minimum, maximum, standard deviation, and skewness of completed order amounts.
> 6. Outlier detection	   : Uses the IQR method to identify unusually low or high order amounts.
> 7. Correlation analysis : Measures relationships between order amount, quantity, and discount percentage.
> 8. Pivot table	       : Summarizes revenue by city and product category, including grand totals.
> 9. Save results         : Exports city revenue, category revenue, and the pivot table as CSV files in the outputs folder.

In [ ]:
from pathlib import Path
import pandas as pd
# Project folder
PROJECT_ROOT = Path(__file__).resolve().parent.parent
# Load cleaned CSV files
orders = pd.read_csv(PROJECT_ROOT / "data" / "clean" / "orders.csv")
customers = pd.read_csv(PROJECT_ROOT / "data" / "clean" / "customers.csv")
products = pd.read_csv(PROJECT_ROOT / "data" / "clean" / "products.csv")
print("=" * 60)
print("INSTAMART CUSTOMER COPILOT - EXPLORATORY DATA ANALYSIS")
print("=" * 60)
# 1. Check the data
print("\n1. DATA SHAPES")
print("Orders:", orders.shape)
print("Customers:", customers.shape)
print("Products:", products.shape)
# 2. Revenue by city
orders_customers = orders.merge(
    customers[["customer_id", "city"]],
    on="customer_id",
    how="left"
)
completed_orders = orders_customers[
    orders_customers["status"].str.lower() == "completed"
].copy()
city_revenue = (
    completed_orders.groupby("city")
    .agg(
        total_revenue=("order_amount", "sum"),
        average_order_amount=("order_amount", "mean"),
        order_count=("order_id", "count")
    )
    .sort_values("total_revenue", ascending=False)
)

In [ ]:
print("\n2. REVENUE BY CITY")
print(city_revenue.round(2).to_string())
# 3. Revenue by product category
orders_products = completed_orders.merge(
    products[["product_id", "category"]],
    on="product_id",
    how="left"
)
category_revenue = (
    orders_products.groupby("category")
    .agg(
        total_revenue=("order_amount", "sum"),
        order_count=("order_id", "count")
    )
    .sort_values("total_revenue", ascending=False)
)
print("\n3. REVENUE BY PRODUCT CATEGORY")
print(category_revenue.round(2).to_string())
# 4. Order amount statistics
amounts = completed_orders["order_amount"]
print("\n4. ORDER AMOUNT STATISTICS")
print(f"Mean: INR {amounts.mean():.2f}")
print(f"Median: INR {amounts.median():.2f}")
print(f"Minimum: INR {amounts.min():.2f}")
print(f"Maximum: INR {amounts.max():.2f}")
print(f"Standard deviation: INR {amounts.std():.2f}")
print(f"Skewness: {amounts.skew():.2f}")
print("\nEDA STEP 1 COMPLETED")
# 5. Outlier detection using the IQR method
print("\n5. OUTLIER DETECTION")
Q1 = amounts.quantile(0.25)

In [ ]:
Q3 = amounts.quantile(0.75)
IQR = Q3 - Q1
lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR
low_outliers = amounts[amounts < lower_limit]
high_outliers = amounts[amounts > upper_limit]
print(f"Q1: INR {Q1:.2f}")
print(f"Q3: INR {Q3:.2f}")
print(f"IQR: INR {IQR:.2f}")
print(f"Lower limit: INR {lower_limit:.2f}")
print(f"Upper limit: INR {upper_limit:.2f}")
print(f"Low outliers: {len(low_outliers)}")
print(f"High outliers: {len(high_outliers)}")
# 6. Correlation analysis
print("\n6. CORRELATION ANALYSIS")
numeric_columns = [
    "order_amount",
    "quantity",
    "discount_pct"
]
correlation_matrix = completed_orders[numeric_columns].corr()
print(correlation_matrix.round(3).to_string())
# 7. Pivot table: revenue by city and category
print("\n7. PIVOT TABLE: CITY VS CATEGORY")
pivot = pd.pivot_table(
    orders_products,
    index="city",
    columns="category",
    values="order_amount",
    aggfunc="sum",
    fill_value=0,
    margins=True,
    margins_name="Grand Total"
)

In [ ]:
print(pivot.round(2).to_string())
print("\nGrand total revenue:")
print(f"INR {pivot.loc['Grand Total', 'Grand Total']:.2f}")
# 8. Save EDA results
output_dir = PROJECT_ROOT / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)
city_revenue.to_csv(output_dir / "revenue_by_city.csv")
category_revenue.to_csv(output_dir / "revenue_by_category.csv")
pivot.to_csv(output_dir / "revenue_pivot_city_category.csv")
print("\n8. EDA RESULTS SAVED")
print("Saved to:", output_dir)